In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from google.colab import files

# Upload CSV File
uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

# Clean Column Names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Column Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

print("\nLast 5 Rows:")
print(df.tail())

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Description:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

# Check Required Columns
required_columns = ["cgpa", "placement_exam_marks", "placed"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing Required Columns:", missing_columns)
else:

    # Missing Value Heatmap
    sns.heatmap(df.isnull(), cbar=False)

    plt.title("Missing Value Heatmap")
    plt.show()

    # Handle Missing Values
    df["cgpa"] = df["cgpa"].fillna(df["cgpa"].median())

    df["placement_exam_marks"] = df[
        "placement_exam_marks"
    ].fillna(
        df["placement_exam_marks"].median()
    )

    df["placed"] = df["placed"].fillna(
        df["placed"].mode()[0]
    )

    print("\nMissing Values After Treatment:")
    print(df.isnull().sum())

    # Placement Mapping
    placement_mapping = {
        0: "Not Placed",
        1: "Placed"
    }

    df["placement_label"] = df["placed"].map(
        placement_mapping
    )

    print("\nPlacement Mapping:")
    print(placement_mapping)

    print(
        df[["placed", "placement_label"]].head()
    )

    # Placement Count
    sns.countplot(x="placed", data=df)

    plt.title("Placement Count")
    plt.xlabel("Placed")
    plt.ylabel("Number of Students")

    plt.show()

    # Placement Status
    sns.countplot(x="placement_label", data=df)

    plt.title("Placement Status")
    plt.xlabel("Placement Status")
    plt.ylabel("Number of Students")

    plt.show()

    print("\nPlacement Counts:")
    print(df["placed"].value_counts())

    # Overall Placement Rate
    placement_rate = df["placed"].mean()

    print("\nOverall Placement Rate:")
    print(placement_rate)

    # Average CGPA by Placement Status
    cgpa_placement = df.groupby(
        "placement_label"
    )["cgpa"].mean()

    print("\nAverage CGPA by Placement Status:")
    print(cgpa_placement)

    sns.barplot(
        x=cgpa_placement.index,
        y=cgpa_placement.values
    )

    plt.title("Average CGPA by Placement Status")
    plt.xlabel("Placement Status")
    plt.ylabel("Average CGPA")

    plt.show()

    # Average Exam Marks by Placement Status
    exam_placement = df.groupby(
        "placement_label"
    )["placement_exam_marks"].mean()

    print(
        "\nAverage Placement Exam Marks by Placement Status:"
    )

    print(exam_placement)

    sns.barplot(
        x=exam_placement.index,
        y=exam_placement.values
    )

    plt.title(
        "Average Placement Exam Marks by Placement Status"
    )

    plt.xlabel("Placement Status")
    plt.ylabel("Average Exam Marks")

    plt.show()

    # CGPA Range Analysis
    df["cgpa_range"] = pd.cut(
        df["cgpa"],
        bins=[0, 6, 7, 8, 9, 10],
        labels=[
            "Below 6",
            "6-7",
            "7-8",
            "8-9",
            "9-10"
        ]
    )

    cgpa_analysis = df.groupby(
        "cgpa_range",
        observed=False
    )["placed"].mean()

    print("\nPlacement Rate by CGPA Range:")
    print(cgpa_analysis)

    sns.barplot(
        x=cgpa_analysis.index,
        y=cgpa_analysis.values
    )

    plt.title("Placement Rate by CGPA Range")
    plt.xlabel("CGPA Range")
    plt.ylabel("Placement Rate")

    plt.ylim(0, 1)

    plt.show()

    # Exam Marks Range Analysis
    df["exam_marks_range"] = pd.cut(
        df["placement_exam_marks"],
        bins=[0, 40, 50, 60, 70, 80, 100],
        labels=[
            "Below 40",
            "40-50",
            "50-60",
            "60-70",
            "70-80",
            "80-100"
        ]
    )

    exam_analysis = df.groupby(
        "exam_marks_range",
        observed=False
    )["placed"].mean()

    print("\nPlacement Rate by Exam Marks:")
    print(exam_analysis)

    sns.barplot(
        x=exam_analysis.index,
        y=exam_analysis.values
    )

    plt.title("Placement Rate by Exam Marks")
    plt.xlabel("Exam Marks Range")
    plt.ylabel("Placement Rate")

    plt.ylim(0, 1)

    plt.xticks(rotation=30)

    plt.show()

    # Correlation Matrix
    correlation = df[
        ["cgpa", "placement_exam_marks", "placed"]
    ].corr()

    print("\nCorrelation Matrix:")
    print(correlation)

    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Correlation Heatmap")

    plt.show()

    # PDF Report
    pdf = PdfPages(
        "placement_eda_report.pdf"
    )

    # Page 1
    fig = plt.figure(
        figsize=(8.27, 11.69)
    )

    plt.subplot(2, 1, 1)

    plt.axis("off")

    plt.text(
        0,
        1,
        "PLACEMENT DATASET EDA REPORT\n\n"
        + "Dataset Shape: "
        + str(df.shape)
        + "\n\nFirst 5 Rows:\n\n"
        + df.head().to_string(index=False),
        fontsize=9,
        verticalalignment="top"
    )

    plt.subplot(2, 1, 2)

    sns.heatmap(
        df[
            [
                "cgpa",
                "placement_exam_marks",
                "placed"
            ]
        ].isnull(),
        cbar=False,
        yticklabels=False
    )

    plt.title("Missing Value Heatmap")

    plt.tight_layout()

    pdf.savefig(fig)

    plt.close()

    # Page 2
    fig = plt.figure(
        figsize=(8.27, 11.69)
    )

    plt.subplot(2, 1, 1)

    sns.countplot(
        x="placement_label",
        data=df
    )

    plt.title("Placement Status")
    plt.xlabel("Placement Status")
    plt.ylabel("Number of Students")

    plt.subplot(2, 1, 2)

    sns.barplot(
        x=cgpa_placement.index,
        y=cgpa_placement.values
    )

    plt.title(
        "Average CGPA by Placement Status"
    )

    plt.xlabel("Placement Status")
    plt.ylabel("Average CGPA")

    plt.tight_layout()

    pdf.savefig(fig)

    plt.close()

    # Page 3
    fig = plt.figure(
        figsize=(8.27, 11.69)
    )

    plt.subplot(2, 1, 1)

    sns.barplot(
        x=exam_placement.index,
        y=exam_placement.values
    )

    plt.title(
        "Average Placement Exam Marks by Placement Status"
    )

    plt.xlabel("Placement Status")
    plt.ylabel("Average Exam Marks")

    plt.subplot(2, 1, 2)

    sns.barplot(
        x=cgpa_analysis.index,
        y=cgpa_analysis.values
    )

    plt.title(
        "Placement Rate by CGPA Range"
    )

    plt.xlabel("CGPA Range")
    plt.ylabel("Placement Rate")

    plt.ylim(0, 1)

    plt.tight_layout()

    pdf.savefig(fig)

    plt.close()

    # Page 4
    fig = plt.figure(
        figsize=(8.27, 11.69)
    )

    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title(
        "Placement Dataset Correlation Heatmap"
    )

    plt.tight_layout()

    pdf.savefig(fig)

    plt.close()

    # Close PDF
    pdf.close()

    print("\nPDF Created Successfully!")

    print("placement_eda_report.pdf")

    # Download PDF
    files.download(
        "placement_eda_report.pdf"
    )

Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset (1).csv
Column Names:
['studentid', 'gender', 'city', 'collegetier', 'stream', 'specialisation', 'hostel', 'historyofbacklogs', 'sgpa_sem1', 'sgpa_sem2', 'sgpa_sem3', 'sgpa_sem4', 'sgpa_sem5', 'sgpa_sem6', 'sgpa_sem7', 'sgpa_sem8', 'cgpa', 'attendancepercent', 'internships', 'projects', 'workshops', 'certifications', 'publications', 'aptitudetestscore', 'softskillsrating', 'codingtestscore', 'mockinterviewscore', 'extracurricular', 'cgpa_tier', 'placementstatus', 'isanomaly', 'salary_package']

First 5 Rows:
   studentid  gender       city collegetier stream specialisation hostel  \
0          1    Male  Ahmedabad       Tier2    ECE     Networking     No   
1          2  Female     Mumbai       Tier2    ECE    DataScience    Yes   
2          3    Male    Kolkata       Tier2     IT    DataScience    Yes   
3          4    Male     Jaipur       Tier1     CS             AI     No   
4          5    Male       Pune 